# Dispatch of a bidding run

Reads a run saved by `uv run bidding run` (`<forecast run>/bidding/<tag>/<timestamp>/`, or the older
`bidding_<tag>/` folders): `report.json`, the water value's decision trace (`decisions.parquet`) and,
per strategy, `backtests/<run>/` with `summary.json`, `dispatch.parquet` (one row per settlement
interval) and `offers.parquet` (every submitted tranche).

For an interactive version: `uv run bidding explore --run <folder>` writes `explore.html` next to it.

In [ ]:
from datetime import datetime, timedelta

import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

from battery_optimisation.bidding.report import list_bidding_runs, load_bidding_report
from battery_optimisation.columns import Col
from battery_optimisation.forecast.report import list_forecast_runs
from battery_optimisation.sim.backtest import RunFiles, StrategyName, load_run

forecast_run = list_forecast_runs()[0]
runs = list_bidding_runs(forecast_run) + sorted(
    (p for p in forecast_run.glob("bidding_*") if (p / "report.json").exists()),
    key=lambda p: (p / "report.json").stat().st_mtime,
    reverse=True,
)
[str(p.relative_to(forecast_run)) for p in runs]

In [ ]:
run_dir = runs[0]  # or e.g. forecast_run / "bidding/market_markov_as_sdpres/latest"
report = load_bidding_report(run_dir)
strategies = {}
for summary_file in (run_dir / "backtests").glob(f"*/{RunFiles.SUMMARY}"):
    summary, frame = load_run(summary_file.parent)
    offers_file = summary_file.parent / RunFiles.OFFERS
    offers = pl.read_parquet(offers_file) if offers_file.exists() else None
    strategies[summary.strategy] = (summary, frame, offers)
decisions = (
    pl.read_parquet(run_dir / "decisions.parquet")
    if (run_dir / "decisions.parquet").exists()
    else None
)
COLORS = {
    StrategyName.WATER_VALUE: "#1f63b5",
    StrategyName.CHRONOS_MEDIAN: "#c0661a",
    StrategyName.PERFECT_FORESIGHT: "#6f7a8a",
}
print(f"{run_dir.name}: {report.start} -> {report.end}, feasibility {report.feasibility}, "
      f"reserve planner {report.bidding.reserve_planner}")
pl.DataFrame(
    [
        {
            "strategy": str(s),
            "profit_sgd": round(t.total.profit_sgd),
            "energy_sgd": round(t.total.revenue_sgd - t.total.degradation_sgd),
            "reserves_sgd": round(t.total.ancillary_sgd),
            "cycles_per_day": round(t.cycles_per_day, 2),
            "engine_cut_mwh": round(t.total.soc_cut_mwh),
            "shortfall_mwh": round(t.total.shortfall_mwh, 1),
        }
        for s, (t, _, _) in strategies.items()
    ]
)

## Profit by month, energy vs reserves

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Energy (SGD)", "Reserves (SGD)"))
for s, (summary, _, _) in strategies.items():
    monthly = pl.DataFrame([m.model_dump() for m in summary.monthly])
    energy = monthly[Col.REVENUE_SGD] - monthly[Col.DEGRADATION_SGD]
    fig.add_bar(x=monthly[Col.PERIOD_START], y=energy, name=str(s), marker_color=COLORS[s],
                legendgroup=str(s), row=1, col=1)
    fig.add_bar(x=monthly[Col.PERIOD_START], y=monthly[Col.ANCILLARY_SGD], name=str(s),
                marker_color=COLORS[s], legendgroup=str(s), showlegend=False, row=1, col=2)
fig.update_layout(barmode="group", height=420, hovermode="x unified")

## Daily gap to perfect foresight

Where the water value loses most against the benchmark: the days to look at first.

In [ ]:
def daily(strategy):
    summary = strategies[strategy][0]
    return pl.DataFrame([d.model_dump() for d in summary.daily]).select(
        Col.PERIOD_START, pl.col(Col.PROFIT_SGD).alias(str(strategy))
    )

gap = (
    daily(StrategyName.PERFECT_FORESIGHT)
    .join(daily(StrategyName.WATER_VALUE), on=Col.PERIOD_START)
    .join(daily(StrategyName.CHRONOS_MEDIAN), on=Col.PERIOD_START)
    .with_columns(
        (pl.col(str(StrategyName.PERFECT_FORESIGHT)) - pl.col(str(StrategyName.WATER_VALUE)))
        .alias("gap_sgd")
    )
)
gap.sort("gap_sgd", descending=True).head(10)

## Dispatch window

Net power, reserves held and SoC for each strategy; set `start` / `days` to look elsewhere
(default: the day with the largest gap).

In [ ]:
start = gap.sort("gap_sgd", descending=True)[0, Col.PERIOD_START]  # or date(2026, 3, 2)
start

In [ ]:
days = 1

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=(
        "USEP (SGD/MWh)",
        "Net power (MW, + discharge)",
        "Reserves held (MW: reg + pri + con)",
        "SoC (MWh)",
    ),
)
first = True
for s, (_, frame, _) in strategies.items():
    w = frame.filter(
        pl.col(Col.INTERVAL_START).dt.date().is_between(start, start + timedelta(days=days - 1))
    )
    if first:
        fig.add_scatter(
            x=w[Col.INTERVAL_START],
            y=w[Col.USEP],
            name="USEP",
            line_shape="hv",
            line_color="#172033",
            row=1,
            col=1,
        )
        first = False
    kw = dict(name=str(s), line_color=COLORS[s], legendgroup=str(s), line_shape="hv")
    fig.add_scatter(x=w[Col.INTERVAL_START], y=w[Col.NET_MW], row=2, col=1, **kw)
    fig.add_scatter(
        x=w[Col.INTERVAL_START],
        y=w[Col.REG_MW] + w[Col.PRI_MW] + w[Col.CON_MW],
        row=3,
        col=1,
        showlegend=False,
        **kw,
    )
    fig.add_scatter(x=w[Col.INTERVAL_START], y=w[Col.SOC_MWH], row=4, col=1, showlegend=False, **kw)
fig.update_layout(height=900, hovermode="x unified")

## One settlement interval

What each strategy offered, what cleared at the realised prices and how it settled. The water
value's decision trace shows what it expected when the offer was fixed (gate closure).

In [ ]:
at = datetime.combine(start, datetime.min.time()).replace(hour=18)  # pick any interval start
at = strategies[StrategyName.WATER_VALUE][1].filter(
    pl.col(Col.INTERVAL_START).dt.replace_time_zone(None) == at
)[0, Col.INTERVAL_START]

settlement = pl.concat(
    [
        frame.filter(pl.col(Col.INTERVAL_START) == at).with_columns(pl.lit(str(s)).alias("strategy"))
        for s, (_, frame, _) in strategies.items()
    ],
    how="diagonal_relaxed",
)
settlement.select("strategy", pl.exclude("strategy"))

In [ ]:
offered = pl.concat(
    [
        offers.filter(pl.col(Col.INTERVAL_START) == at).with_columns(pl.lit(str(s)).alias("strategy"))
        for s, (_, _, offers) in strategies.items()
        if offers is not None
    ]
)
usep = settlement[0, Col.USEP]
offered.with_columns(
    pl.when(pl.col(Col.PRODUCT) != "energy").then(None)
    .when(pl.col(Col.SIDE) == "discharge").then(pl.col(Col.OFFER_PRICE) <= usep)
    .otherwise(pl.col(Col.OFFER_PRICE) >= usep)
    .alias("energy_clears")
)

In [ ]:
if decisions is not None:
    print(decisions.filter(pl.col(Col.INTERVAL_START) == at).to_dicts())